# 3. Advanced Data Tools

**Goal:** Empower the agent to query enterprise data warehouses (like Snowflake) to answer factual business questions.

**Key Concept:**
We use the **Model Context Protocol (MCP)** to connect our agent to a DataRobot deployment acting as a secure "Data Tool." This setup enables the agent to dynamically generate queries and retrieve live datasets—transforming it from a simple chatbot into a data analyst capable of answering questions like *"What distinct bakeries are we tracking supplies for?"*

In [ ]:
import os
from dotenv import load_dotenv
from pprint import pprint
import datarobot as dr
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.mcp import MCPServerStreamableHTTP

load_dotenv()
dr_client = dr.Client()

MCP_DEPLOYMENT_ID = os.getenv("MCP_DEPLOYMENT_ID")

server = MCPServerStreamableHTTP(
    f"{dr_client.endpoint}/deployments/{MCP_DEPLOYMENT_ID}/directAccess/mcp",
    headers={
        "Authorization": f"Bearer {dr_client.token}",
        "x-datarobot-api-token": dr_client.token,
    },
    timeout=60.0,
)

MODEL_NAME = os.getenv("MODEL_NAME", "azure/gpt-5-2025-08-07")
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        api_key=dr_client.token, base_url=dr_client.endpoint + "/genai/llmgw"
    ),
)

system_prompt = """
Use the ERCOT dataset, available in the DataRobot AI catalog, to answer user questions.
"""

agent = Agent(model=model, toolsets=[server], system_prompt=system_prompt)

async with server:
    response = await agent.run("What is the average dam_price_usd_mwh for all hubs?")
    pprint(response.output)

In [ ]:
DATASET_ID = os.getenv("ERCOT_TRAINING_DATASET_ID")

async with server:
    response = await agent.run(f"How many unique hub_name are in dataset {DATASET_ID}?")
    pprint(response.output)